# 🔄 Day 2 — Cross-Validation

**Week 4 — Model Evaluation & Selection**

Today we move from a **single validation split** to **cross-validation**, giving us a more reliable view of how consistently a model performs.

> 🎯 **Main Goal:** Evaluate a classification model across multiple folds and interpret the **mean ± standard deviation** without touching the final test set.

## 🎯 Learning Objectives

By the end of this notebook, I should be able to:

1. Explain why cross-validation is more reliable than a single validation split.
2. Explain how **k-fold cross-validation** works.
3. Use Scikit-learn's `cross_val_score`.
4. Calculate and interpret the **mean** and **standard deviation** of cross-validation scores.
5. Explain why **Stratified K-Fold** matters for classification.
6. Compare a cross-validated estimate with a single-split result.
7. Keep the final **test set untouched** during model development.

## 📚 Topics Covered

- ⚠️ Limitations of a single validation split
- 🔄 k-fold cross-validation
- 🔢 Choosing the number of folds
- 🧪 `cross_val_score`
- 📊 Mean and standard deviation
- ⚖️ Stratified K-Fold for classification
- 🆚 Cross-validation vs. a single split
- 🔒 Keeping the test set untouched

## 💡 Introduction

In **Day 1**, we learned that a dataset should be separated into **training, validation, and test sets**. The validation set helps us make development decisions without using the final test set.

However, a single validation set has a weakness: our evaluation can depend heavily on the particular observations that happened to be placed in that validation set.

Imagine that one validation set contains mostly easy examples. The model may receive a very high score. Another random split may contain more difficult examples and produce a lower score.

So we have a new question:

> **What if our validation score is simply the result of getting a lucky or unlucky split?**

Cross-validation addresses this problem by repeating the training and validation process across multiple folds.

Instead of trusting one validation score, we obtain several scores and summarize them using:

- 📈 **Mean:** average performance across folds.
- 📏 **Standard deviation:** how much performance varies across folds.

For classification, we also need to make sure that the folds have a reasonable representation of each class. This is where **Stratified K-Fold** becomes important.

In this notebook, we will apply these ideas to the **Titanic dataset from the uploaded file** and use Logistic Regression as our classification model.

## 🚢 Dataset — Titanic

We will work directly with the supplied **`titanic_train.csv`** dataset.

Our target variable is:

- 🎯 `Survived` — whether the passenger survived.

We will use the following numerical features available in the dataset:

- `Pclass`
- `Age`
- `SibSp`
- `Parch`
- `Fare`

The purpose of this notebook is to focus on **cross-validation**, so we will keep the feature set simple and avoid introducing new feature engineering.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

RANDOM_STATE = 42

# Load the supplied Titanic dataset
data = pd.read_csv("titanic_train.csv")

print("Dataset shape:", data.shape)
data.head()

Dataset shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 🔍 2.1 Inspect the Dataset

Before modeling, we need to understand the structure of the data.

We will check:

- dataset dimensions,
- column types,
- missing values,
- target distribution.

In [ ]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [ ]:
print("Missing values:")
print(data.isnull().sum())

Missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [ ]:
print("Target distribution:")
print(data["Survived"].value_counts())

print("\nTarget proportions:")
print(data["Survived"].value_counts(normalize=True))

Target distribution:
Survived
0    549
1    342
Name: count, dtype: int64

Target proportions:
Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64


### 🧠 Interpretation

`Survived` is a binary classification target.

The target distribution tells us how common each class is. Because the proportions of the classes matter when evaluating a classifier, we want our cross-validation folds to maintain a similar distribution.

This is one of the main reasons we use **Stratified K-Fold** for classification.

## 🧹 2.2 Prepare Features and Target

We will use these features:

- `Pclass`
- `Age`
- `SibSp`
- `Parch`
- `Fare`

`Age` and possibly other numerical columns can contain missing values. For this exercise, we will fill missing numerical values using the median.

> ⚠️ **Important:** This simple imputation is used here for learning the cross-validation concept. In later pipeline-focused work, preprocessing should be placed inside a Scikit-learn `Pipeline` so that preprocessing is fitted only on the training portion of each fold.
**Note on scope:** `Sex` and `Embarked` are deliberately left out here, even though `Sex` is the
single strongest predictor of survival on Titanic. This notebook's goal is to isolate and learn
the cross-validation mechanics (folds, mean ± std, stratification) on a small, purely numeric
feature set — not to build the strongest possible classifier. Because of this, the F1 scores
below (~0.49–0.55) are noticeably lower than they would be with `Sex` included; the pattern in
the scores (their spread, the mean vs. the test score) is still fully valid and is the point of
this notebook, even though the absolute score is not competitive.


In [ ]:
features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]

X = data[features].copy()
y = data["Survived"].copy()

# Simple preparation for this learning exercise
X = X.fillna(X.median())

print("Features:", features)
print("X shape:", X.shape)
print("y shape:", y.shape)

Features: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
X shape: (891, 5)
y shape: (891,)


## 🔒 2.3 Hold Out the Final Test Set

Cross-validation should be performed on the **training data**, not on the final test set.

We first hold out 20% of the data as the final test set.

The test set will not participate in cross-validation or model development decisions.

### Workflow

**Full Dataset**  
↓  
**Training Data (80%)** → Cross-Validation  
**Test Data (20%)** → Final evaluation only

This preserves the purpose of the test set as an honest estimate of performance on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (712, 5)
Test set: (179, 5)


## 🤖 2.4 Define the Model

We will use **Logistic Regression**, which we already worked with during Week 3.

The model stays fixed because the purpose of Day 2 is to learn how to evaluate it using cross-validation.

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

model

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

## 🔄 2.5 What Does k-Fold Cross-Validation Do?

In **k-fold cross-validation**, the training data is divided into `k` folds.

For **5-fold cross-validation**:

| Round | Training | Validation |
|---|---|---|
| 1 | Folds 2–5 | Fold 1 |
| 2 | Folds 1, 3–5 | Fold 2 |
| 3 | Folds 1–2, 4–5 | Fold 3 |
| 4 | Folds 1–3, 5 | Fold 4 |
| 5 | Folds 1–4 | Fold 5 |

Every observation is used:

- ✅ once for validation,
- ✅ four times for training.

This gives us five performance estimates instead of one.

## 📊 2.6 Why Mean ± Standard Deviation?

After cross-validation, we obtain one score for every fold.

For example:

`[0.71, 0.74, 0.69, 0.73, 0.72]`

We can summarize these scores as:

**Mean ± Standard Deviation**

- 📈 **Mean:** typical model performance.
- 📏 **Standard deviation:** variability across folds.

A high mean with a small standard deviation generally indicates strong and consistent performance.

A high mean with a large standard deviation means the model performs well on average but is less consistent across different subsets of the data.

## 🧪 2.7 Run 5-Fold Cross-Validation

Scikit-learn's `cross_val_score` makes it easy to run cross-validation.

We will:

1. use the training data only,
2. use 5 folds,
3. evaluate using F1-score,
4. inspect the score from every fold.

In [ ]:
cv_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="f1"
)

print("F1-score for each fold:")
print(cv_scores)

F1-score for each fold:
[0.53932584 0.54945055 0.53763441 0.55813953 0.57446809]


In [ ]:
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

print(f"Mean F1-score: {cv_mean:.4f}")
print(f"Standard deviation: {cv_std:.4f}")
print(f"Performance: {cv_mean:.4f} ± {cv_std:.4f}")

Mean F1-score: 0.5518
Standard deviation: 0.0135
Performance: 0.5518 ± 0.0135


## 🧠 2.8 Interpret the Cross-Validation Results

The five folds do not necessarily produce identical scores because each fold contains different observations.

The **mean F1-score** gives us the average performance across the five folds.

The **standard deviation** tells us how much that performance changes from one fold to another.

### What should I look for?

- If the standard deviation is relatively small → performance is more consistent.
- If the standard deviation is relatively large → performance varies more between folds.

The important point is that cross-validation gives us more information than a single score.

## ⚖️ 2.9 Stratified K-Fold

For classification, we do not want one fold to contain a very different proportion of classes from the rest of the dataset.

**Stratified K-Fold** solves this by preserving the class distribution across folds.

Scikit-learn automatically uses a stratified strategy for classifiers when `cv=5` is supplied to `cross_val_score`.

However, we will explicitly create a `StratifiedKFold` object so that the process is visible and reproducible.

In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

stratified_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=skf,
    scoring="f1"
)

print("Stratified 5-fold F1-scores:")
print(stratified_scores)

print(f"\nMean: {stratified_scores.mean():.4f}")
print(f"Std: {stratified_scores.std():.4f}")

Stratified 5-fold F1-scores:
[0.55813953 0.53763441 0.5625     0.46153846 0.58823529]

Mean: 0.5416
Std: 0.0432


## 🔎 2.10 Verify Stratification

Let's inspect the survival proportion in each validation fold.

If stratification is working correctly, the class proportions should remain reasonably close to the overall training-data proportion.

In [ ]:
overall_survival_rate = y_train.mean()

print(f"Overall training survival rate: {overall_survival_rate:.3f}\n")

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_train, y_train),
    start=1
):
    fold_survival_rate = y_train.iloc[val_idx].mean()
    print(
        f"Fold {fold} validation survival rate: "
        f"{fold_survival_rate:.3f}"
    )

Overall training survival rate: 0.383



Fold 1 validation survival rate: 0.385
Fold 2 validation survival rate: 0.385
Fold 3 validation survival rate: 0.387
Fold 4 validation survival rate: 0.380
Fold 5 validation survival rate: 0.380


### 💡 Interpretation

The validation folds have similar survival proportions.

This is the purpose of stratification: each fold remains reasonably representative of the original class distribution.

For classification problems, especially when classes are imbalanced, this helps make model evaluation more reliable.

## 🆚 2.11 Cross-Validation vs. a Single Validation Split

A single validation split gives us one performance estimate.

Cross-validation gives us several estimates and summarizes them.

| Method | Number of validation evaluations | Information |
|---|---:|---|
| Single split | 1 | One performance estimate |
| 5-fold CV | 5 | Mean + variability |

Cross-validation does **not** guarantee a higher score.

Its main advantage is that our conclusion is less dependent on one particular split.

## 🧪 2.12 Single-Split Comparison

To compare fairly, we will use the same metric — **F1-score**.

The test set is still reserved for the final evaluation. We will first calculate the cross-validation result, then train the final model on all training data and evaluate it on the untouched test set.

In [ ]:
# Train the final model on the full training set
model.fit(X_train, y_train)

# Final evaluation on the untouched test set
y_test_pred = model.predict(X_test)

test_f1 = f1_score(y_test, y_test_pred)

print(f"Cross-validation mean F1: {cv_mean:.4f}")
print(f"Cross-validation std F1:  {cv_std:.4f}")
print(f"Final test F1:            {test_f1:.4f}")

Cross-validation mean F1: 0.5518


Cross-validation std F1:  0.0135
Final test F1:            0.4912


## 🧠 2.13 Interpretation of the Comparison

The cross-validation mean estimates how the model performs across multiple validation folds of the training data.

The final test score answers a different question:

> **How well does the finalized model perform on data that was kept completely separate during development?**

The two values can be similar or different. A difference does not automatically mean something is wrong.

The important discipline is:

- 🔄 Use cross-validation during development.
- 🔒 Keep the test set untouched.
- 🏁 Use the test set only for the final evaluation.

## 🚫 2.14 Why We Must Not Tune on the Test Set

Suppose we repeatedly try different models or hyperparameters and keep choosing the one with the best test score.

Even though the model never directly trains on the test observations, our **decisions** are being influenced by test performance.

Over time, the test set effectively becomes part of the development process.

That means the final test score is no longer a truly independent estimate of generalization.

### Correct workflow

**Training data** → Cross-validation → Model decisions → Final model → **Test set once**

### Incorrect workflow

**Training data** → Test score → Change model → Test score again → Change model → Test score again

The second workflow leaks information from the test set into the modeling process.

# 🖥️ Hands-On Lab

### Step 1 — Run 5-Fold Cross-Validation

Evaluate the Week 3 Logistic Regression model using `cross_val_score`.

### Step 2 — Report Mean ± Standard Deviation

Calculate and report the mean and standard deviation.

### Step 3 — Compare Results

Compare the cross-validation mean with the final test score.

### Step 4 — Explain Stratification

Explain why stratified folds are appropriate for this Titanic classification problem.

In [ ]:
# Lab Step 1 + Step 2

lab_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=skf,
    scoring="f1"
)

lab_mean = lab_scores.mean()
lab_std = lab_scores.std()

print("Fold scores:", lab_scores)
print(f"Mean ± Std: {lab_mean:.4f} ± {lab_std:.4f}")

Fold scores: [0.55813953 0.53763441 0.5625     0.46153846 0.58823529]
Mean ± Std: 0.5416 ± 0.0432


In [ ]:
# Lab Step 3

print(f"Cross-validation mean F1: {lab_mean:.4f}")
print(f"Cross-validation std F1:  {lab_std:.4f}")
print(f"Final test F1:            {test_f1:.4f}")

Cross-validation mean F1: 0.5416
Cross-validation std F1:  0.0432
Final test F1:            0.4912


### ✍️ Lab Step 4 — Written Explanation

**Why does Stratified K-Fold matter here?**

Titanic survival is a classification problem with two classes, and those classes aren't evenly
split — about 38% of passengers survived and 62% did not. If the 5 folds were created without
stratification, plain random chance could put unusually few (or unusually many) survivors into
one particular fold. That fold's F1 score would then reflect an easier or harder validation set
rather than the model's true performance, and it could pull the overall mean up or down for
reasons that have nothing to do with the model itself.

Stratified K-Fold fixes this by preserving the ~38% survival rate in every fold, confirmed
directly in Section 2.10 above (each fold's validation survival rate landed between 0.380 and
0.387, right around the training set's overall 0.383). That makes every fold a fair, representative
test of the model, so the mean and standard deviation reported in Section 2.9 can be trusted as an
honest summary of performance rather than an artifact of how the data happened to be shuffled.


# 📌 Key Findings

- 🔄 Cross-validation evaluates a model across multiple train-validation splits.
- 📊 The mean summarizes average performance across folds.
- 📏 The standard deviation shows how much performance varies across folds.
- ⚖️ Stratified K-Fold preserves class proportions for classification.
- 🔒 The final test set must remain untouched during development.
- 🧪 Cross-validation is used for model evaluation and development decisions.
- 🏁 The final test score is reserved for the finalized model.
- 💡 Cross-validation makes our **performance estimate more reliable**; it does not automatically make the model better.

# 🪞 Reflection

### What did I learn?

Cross-validation showed me why a single validation split can be misleading. A model can receive a different score depending on which observations happen to be selected for validation.

By using multiple folds, I can evaluate the model on different subsets of the training data and obtain both an average performance estimate and a measure of stability.

I also learned why stratification is important for classification: it helps maintain a similar class distribution across the validation folds.

### Main Takeaway

> **Cross-validation does not make the model better; it makes our estimate of its performance more reliable.**

# 🏁 Conclusion

In this notebook, I applied **5-fold cross-validation** to the Titanic classification dataset using Logistic Regression.

I used Scikit-learn's `cross_val_score` to obtain a score for each fold, then summarized the results using the **mean and standard deviation**.

I also used **Stratified K-Fold** to preserve the class distribution across folds and compared the cross-validation estimate with the final score on the untouched test set.

The main lesson from Day 2 is that model evaluation should not depend on a single lucky or unlucky validation split. Cross-validation gives us a more stable and informative estimate during development while preserving the final test set for an honest evaluation.

# 🛠️ Tools Used

- 🐍 **Python**
- 🐼 **Pandas**
- 🔢 **NumPy**
- 🤖 **Scikit-learn**
  - `train_test_split`
  - `cross_val_score`
  - `StratifiedKFold`
  - `LogisticRegression`
  - `f1_score`
- 📓 **Jupyter Notebook / Google Colab**

# 📤 GitHub Submission

**Notebook:** `Week4_Day2_Cross_Validation_Titanic.ipynb`

The notebook includes:

- 📚 Concept explanation
- 🚢 Titanic dataset
- 🔄 5-fold cross-validation
- 📊 Mean ± standard deviation
- ⚖️ Stratified K-Fold
- 🆚 Cross-validation vs. single-split evaluation
- 🔒 Test-set discipline
- 🖥️ Hands-On Lab
- 🪞 Reflection
- 🏁 Conclusion